# Queue Cure: RandomForestRegressor Demo

This notebook trains a Random Forest regression model to predict hospital patient waiting time. It can be opened in Jupyter Notebook or uploaded directly to Google Colab.

In [ ]:
# Run this cell in Google Colab if the packages are not installed.
%pip install -q pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import plot_tree

RANDOM_SEED = 42

DOCTORS = [
    (1, "General Medicine", 10, 1.0),
    (2, "General Medicine", 12, 1.1),
    (3, "Pediatrics", 8, 0.9),
    (4, "Orthopedics", 15, 1.15),
    (5, "Dermatology", 7, 0.85),
    (6, "Cardiology", 18, 1.2),
]

SLOW_HOURS = {12, 13, 17, 18}
FAST_HOURS = {9, 10, 15}

def generate_dataset(n_rows=6000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    doctor_ids = rng.choice([doctor[0] for doctor in DOCTORS], size=n_rows)
    doctor_lookup = {doctor[0]: doctor for doctor in DOCTORS}
    rows = []

    for doctor_id in doctor_ids:
        _, department, base_consult, efficiency = doctor_lookup[doctor_id]
        patients_ahead = int(rng.integers(0, 16))
        avg_consultation_time = max(3, int(rng.normal(base_consult, 2)))
        current_hour = int(rng.integers(8, 20))
        day_of_week = int(rng.integers(1, 8))
        total_active_patients = patients_ahead + int(rng.integers(0, 10))

        base_wait = patients_ahead * avg_consultation_time * efficiency
        if current_hour in SLOW_HOURS:
            base_wait *= 1.25
        elif current_hour in FAST_HOURS:
            base_wait *= 0.9
        if day_of_week in (6, 7):
            base_wait *= 0.8

        base_wait *= 1 + (total_active_patients / 100)
        noise = rng.normal(0, max(2.0, base_wait * 0.08))

        rows.append({
            "patientsAhead": patients_ahead,
            "avgConsultationTime": avg_consultation_time,
            "doctorId": doctor_id,
            "department": department,
            "currentHour": current_hour,
            "dayOfWeek": day_of_week,
            "totalActivePatients": total_active_patients,
            "waitTime": round(max(0, base_wait + noise), 1),
        })

    return pd.DataFrame(rows)

In [ ]:
df = generate_dataset()
+display(df.head())
+print(f"Dataset shape: {df.shape}")

## Prepare the training data

`department` is categorical, so it is converted into one-hot encoded columns. The target is the continuous `waitTime` value.

In [ ]:
numeric_features = [
+    "patientsAhead", "avgConsultationTime", "doctorId",
+    "currentHour", "dayOfWeek", "totalActivePatients"
+]
+
+try:
+    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
+except TypeError:  # Compatibility with older scikit-learn versions.
+    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
+
+X_numeric = df[numeric_features].to_numpy(dtype=float)
+X_department = encoder.fit_transform(df[["department"]])
+X = np.hstack([X_numeric, X_department])
+y = df["waitTime"].to_numpy()
+
+feature_names = numeric_features + list(encoder.get_feature_names_out(["department"]))
+
+X_train, X_test, y_train, y_test = train_test_split(
+    X, y, test_size=0.2, random_state=RANDOM_SEED
+)
+
+print(f"Training rows: {len(X_train)}")
+print(f"Testing rows: {len(X_test)}")

## Train and evaluate the Random Forest

The forest contains 150 decision trees. The final prediction is the average prediction of those trees.

In [ ]:
model = RandomForestRegressor(
+    n_estimators=150,
+    max_depth=12,
+    min_samples_leaf=4,
+    random_state=RANDOM_SEED,
+    n_jobs=-1,
+)
+
+model.fit(X_train, y_train)
+predictions = model.predict(X_test)
+
+mae = mean_absolute_error(y_test, predictions)
+r2 = r2_score(y_test, predictions)
+
+print(f"Number of trees: {len(model.estimators_)}")
+print(f"Validation MAE: {mae:.2f} minutes")
+print(f"Validation R²: {r2:.3f}")

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_names)
+importance = importance.sort_values(ascending=True)
+
+plt.figure(figsize=(9, 5))
+importance.plot(kind="barh")
+plt.xlabel("Importance")
+plt.title("Random Forest Feature Importance")
+plt.show()

In [ ]:
plt.figure(figsize=(24, 10))
+plot_tree(
+    model.estimators_[0],
+    feature_names=feature_names,
+    filled=True,
+    max_depth=3,
+    rounded=True,
+    fontsize=8,
+)
+plt.title("One Decision Tree from the Random Forest")
+plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
+plt.scatter(y_test, predictions, alpha=0.5)
+minimum = min(y_test.min(), predictions.min())
+maximum = max(y_test.max(), predictions.max())
+plt.plot([minimum, maximum], [minimum, maximum], "r--", label="Perfect prediction")
+plt.xlabel("Actual waiting time (minutes)")
+plt.ylabel("Predicted waiting time (minutes)")
+plt.title("Actual vs Predicted Waiting Time")
+plt.legend()
+plt.show()

## Predict a new patient

In [ ]:
new_patient = pd.DataFrame([{
+    "patientsAhead": 8,
+    "avgConsultationTime": 10,
+    "doctorId": 1,
+    "department": "General Medicine",
+    "currentHour": 11,
+    "dayOfWeek": 3,
+    "totalActivePatients": 12,
+}])
+
+new_numeric = new_patient[numeric_features].to_numpy(dtype=float)
+new_department = encoder.transform(new_patient[["department"]])
+new_X = np.hstack([new_numeric, new_department])
+
+predicted_wait = model.predict(new_X)[0]
+print(f"Predicted waiting time: {predicted_wait:.1f} minutes")